# Bag of Words e TF-IDF

In questo notebook esploreremo come rappresentare i testi come vettori numerici utilizzando gli approcci Bag of Words (BoW) e TF-IDF (Term Frequency - Inverse Document Frequency).

# 🎒 Bag of Words (BoW): Il Testo Diventa Matematica

Dopo aver pulito il nostro testo, ci scontriamo con la realtà: **i modelli di Machine Learning non leggono parole, leggono numeri.** Dobbiamo quindi trovare un modo per trasformare le stringhe in vettori numerici. Questo processo si chiama **Vettorizzazione**.

Il metodo più semplice e intuitivo per farlo è il modello **Bag of Words**.

---

### 1. Il Concetto: Cos'è una "Borsa di Parole"?

Immaginate di prendere i vostri documenti, ritagliare ogni singola parola e gettarla dentro un sacchetto. In questa operazione:
1.  **Perdiamo l'ordine delle parole:** Non sappiamo più chi viene prima e chi dopo.
2.  **Perdiamo la grammatica:** Non ci interessa la struttura della frase.
3.  **Manteniamo la frequenza:** Ci interessa solo **quali** parole sono presenti e **quante volte** compaiono.

![Concetto Bag of Words](img/bow.png)

---

### 2. Come funziona (I 3 Step della Vettorizzazione)

Per trasformare un insieme di documenti (**Corpus**) in numeri, seguiamo questi passaggi:

1.  **Costruzione del Vocabolario:** Identifichiamo tutte le parole uniche presenti nell'intero corpus (es. se abbiamo 1000 libri, il vocabolario potrebbe avere 50.000 parole uniche).
2.  **Vettorizzazione:** Ogni documento diventa un vettore lungo quanto il vocabolario. 
3.  **Conteggio:** In ogni posizione del vettore, inseriamo il numero di volte che quella specifica parola appare nel documento.

### 3. La Document-Term Matrix (DTM)

Prendiamo due frasi d'esempio:
* *Doc 1:* "Il gatto mangia."
* *Doc 2:* "Il cane mangia."

Il nostro vocabolario sarà: `[il, gatto, mangia, cane]`. 
La rappresentazione numerica sarà una tabella (matrice) dove ogni riga è un documento e ogni colonna è una parola del vocabolario:

| Documento | il | gatto | mangia | cane |
| :--- | :---: | :---: | :---: | :---: |
| **Doc 1** | 1 | 1 | 1 | 0 |
| **Doc 2** | 1 | 0 | 1 | 1 |

In termini formali, ogni documento $d$ viene rappresentato come un vettore $v$:
$$v_d = [f_1, f_2, ..., f_n]$$
dove $f_n$ è la frequenza della parola $n$-esima del vocabolario nel documento specifico.

---


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

# 1. Definiamo i nostri "documenti giocattolo" (Toy Dataset)
# Immaginiamo siano frasi già passate per il preprocessing
corpus = [
    "il gatto mangia il topo",
    "il cane mangia l osso",
    "il topo scappa dal gatto", #aggiungi esempi
]

print("--- CORPUS DI PARTENZA ---")
for i, doc in enumerate(corpus):
    print(f"Doc {i}: {doc}")

In [25]:
# 2. Inizializziamo il CountVectorizer
# Questo oggetto si occuperà di creare il vocabolario e contare le frequenze
vectorizer = CountVectorizer()


In [26]:
# 3. Fit & Transform
# 'fit' impara il vocabolario; 'transform' crea la matrice dei conteggi
X = vectorizer.fit_transform(corpus)

In [27]:
# 4. Vediamo il vocabolario creato (le nostre "colonne")
# Notate come le parole sono ordinate alfabeticamente
vocabolario = vectorizer.get_feature_names_out()
print(f"\n--- VOCABOLARIO (Parole Uniche) ---\n{vocabolario}")


--- VOCABOLARIO (Parole Uniche) ---
['cane' 'che' 'dal' 'fare' 'gatto' 'il' 'lezione' 'mangia' 'oggi' 'osso'
 'palle' 'scappa' 'sono' 'topo' 'venuto']


In [28]:
# 5. Trasformiamo la matrice sparsa in un DataFrame per una visualizzazione leggibile
# Ogni riga è un documento, ogni colonna è una parola del vocabolario
bow_matrix = pd.DataFrame(X.toarray(), columns=vocabolario)

print("\n--- DOCUMENT-TERM MATRIX (Bag of Words) ---")
print(bow_matrix)


--- DOCUMENT-TERM MATRIX (Bag of Words) ---
   cane  che  dal  fare  gatto  il  lezione  mangia  oggi  osso  palle  \
0     0    0    0     0      1   2        0       1     0     0      0   
1     1    0    0     0      0   1        0       1     0     1      0   
2     0    0    1     0      1   1        0       0     0     0      0   
3     0    1    0     1      0   0        1       0     1     0      1   
4     0    1    0     1      0   0        1       0     1     0      1   

   scappa  sono  topo  venuto  
0       0     0     1       0  
1       0     0     0       0  
2       1     0     1       0  
3       0     1     0       1  
4       0     1     0       1  


In [29]:
# 6. Analizziamo un dettaglio per gli studenti
print("\n--- ANALISI VELOCE ---")
print(f"La parola 'mangia' appare nei documenti 0 e 1? \n{bow_matrix['palle']}")


--- ANALISI VELOCE ---
La parola 'mangia' appare nei documenti 0 e 1? 
0    0
1    0
2    0
3    1
4    1
Name: palle, dtype: int64


### ⚠️ I limiti del Bag of Words
Prima di passare al codice, è fondamentale capire cosa stiamo sacrificando:

* **Sparsità:** Se il vocabolario ha 50.000 parole e un tweet ne ha solo 10, il vettore sarà pieno di zeri (uno spreco di memoria enorme!).
* **Perdita di Semantica:** Per il BoW, *"Il cane morde l'uomo"* e *"L'uomo morde il cane"* producono **lo stesso identico vettore**, anche se il significato è molto diverso.
* **Peso delle parole:** Le parole molto comuni (es. "anche", "solo") avranno conteggi alti ma scarso valore informativo per distinguere i documenti.

> **Spoiler:** Per risolvere l'ultimo punto, vedremo presto come il **TF-IDF** assegna un peso più intelligente alle parole rare e significative!

# ⚖️ TF-IDF: Pesare l'Informanza delle Parole

Il modello **Bag of Words** che abbiamo visto ha un limite enorme: tratta tutte le parole allo stesso modo. In una collezione di articoli di medicina, la parola *"il"* apparirà migliaia di volte, esattamente come la parola *"paziente"*. Ma quale delle due ci aiuta davvero a capire di cosa parla un testo?

Il **TF-IDF** (Term Frequency - Inverse Document Frequency) nasce per risolvere questo problema, dando più importanza alle parole **rare e specifiche** e "punendo" le parole troppo comuni.

---

### 1. L'Intuizione: Frequenza vs Rarità

L'idea è che il valore di una parola dipenda da un equilibrio tra due forze:

1.  **TF (Term Frequency):** "Se una parola appare molte volte in questo documento, deve essere importante *per questo* documento."
2.  **IDF (Inverse Document Frequency):** "Se una parola appare in quasi tutti i documenti della mia collezione, allora è una parola generica (rumore) e non è utile per distinguere questo testo dagli altri."



---

### 2. Esempio Pratico: Gli "Elementi Distintivi"

Immaginiamo di avere un database di 1.000 articoli. Analizziamo tre parole in un articolo che parla di **Esplorazione Spaziale**:

| Parola | Frequenza nel Documento (TF) | Diffusione nel Corpus (IDF) | Risultato TF-IDF | Perché? |
| :--- | :--- | :--- | :--- | :--- |
| **"il"** | ⭐⭐⭐⭐⭐ (Altissima) | 🌍🌍🌍 (In tutti i 1.000 doc) | 📉 **Quasi Zero** | È ovunque, non dà informazione. |
| **"lancio"** | ⭐⭐⭐ (Media) | 🌍 (In 50 documenti) | 📈 **Alto** | È specifica del tema spaziale. |
| **"esopianeta"** | ⭐⭐ (Bassa) | 🌑 (In soli 2 documenti) | 🚀 **Massimo** | È rarissima e identifica univocamente il tema. |

---

### 3. La Formula (Senza Paura)

Il punteggio TF-IDF è il prodotto di due componenti:

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

* **TF (Term Frequency):** $\frac{\text{Conteggio parola nel Doc}}{\text{Totale parole nel Doc}}$
* **IDF (Inverse Document Frequency):** $\log\left( \frac{\text{Totale Documenti}}{\text{Documenti che contengono la parola}} \right)$

> **Cosa succede matematicamente?**
> * Se una parola è presente in **tutti** i documenti, il rapporto nel logaritmo diventa 1. Poiché $\log(1) = 0$, il peso della parola viene annullato.
> * Più una parola è rara (pochi documenti la contengono), più il valore del logaritmo cresce, "esaltando" il punteggio finale.

---

### 4. Perché è meglio del Bag of Words?

* **Rilevanza:** Identifica automaticamente le "Keyword" di un testo.
* **Pulizia Automatica:** Riduce l'impatto delle stop-word che potrebbero essere sfuggite al preprocessing.
* **Confronto:** Due documenti sono simili non solo se hanno parole in comune, ma se hanno **parole importanti** in comune.

In [58]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Corpus di esempio
# Notate come la parola "IA" sia presente ovunque, 
# mentre "etica" o "musica" siano specifiche.
corpus = [
    "l IA è il futuro dell informatica",
    "l etica nell IA è un tema cruciale",
    "generare musica con l IA è divertente"


]

# 2. Inizializziamo il TfidfVectorizer
# Usiamo parametri di default per ora
tfidf_vectorizer = TfidfVectorizer()

In [59]:
# 3. Fit & Transform
# Calcola TF e IDF contemporaneamente
X_tfidf = tfidf_vectorizer.fit_transform(corpus)

In [60]:
# 4. Recuperiamo i nomi delle colonne (il vocabolario)
vocabolario = tfidf_vectorizer.get_feature_names_out()

In [61]:
# 5. Creiamo un DataFrame per visualizzare i pesi
tfidf_matrix = pd.DataFrame(X_tfidf.toarray(), columns=vocabolario)

print("--- MATRICE TF-IDF ---")
print(tfidf_matrix.round(3)) # Arrotondiamo a 3 decimali per leggibilità

--- MATRICE TF-IDF ---
    con  cruciale  dell  divertente  etica  futuro  generare     ia    il  \
0  0.00     0.000  0.48        0.00  0.000    0.48      0.00  0.283  0.48   
1  0.00     0.432  0.00        0.00  0.432    0.00      0.00  0.255  0.00   
2  0.48     0.000  0.00        0.48  0.000    0.00      0.48  0.283  0.00   

   informatica  musica   nell   tema     un  
0         0.48    0.00  0.000  0.000  0.000  
1         0.00    0.00  0.432  0.432  0.432  
2         0.00    0.48  0.000  0.000  0.000  


In [62]:
# 6. Confronto tra una parola comune e una rara
print("\n--- ANALISI DEI PESI ---")
parola_comune = "ia"
parola_rara = "etica"

print(f"Peso di '{parola_comune}' nel Doc 1: {tfidf_matrix.loc[1, parola_comune]}")
print(f"Peso di '{parola_rara}' nel Doc 1: {tfidf_matrix.loc[1, parola_rara]}")


--- ANALISI DEI PESI ---
Peso di 'ia' nel Doc 1: 0.25537359879528915
Peso di 'etica' nel Doc 1: 0.43238508878969045


# 🏆 Case Study: Sentiment Analysis - BoW vs TF-IDF

In questa sezione finale, metteremo alla prova le due rappresentazioni che abbiamo studiato. Costruiremo un classificatore di **Sentiment Analysis** per distinguere recensioni di film positive da quelle negative.

### Il Piano d'Azione:
1. **Caricamento dati**: Useremo il dataset `movie_reviews` di NLTK.
2. **Preprocessing**: Pulizia profonda di tutti i testi.
3. **Feature Extraction**: Creeremo due versioni dei dati (BoW e TF-IDF).
4. **Training**: Alleneremo un modello di **Regressione Logistica** su entrambi.
5. **Evaluation**: Confronteremo l'accuratezza per decretare il vincitore.

In [132]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import movie_reviews
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from nltk.stem import PorterStemmer

# ==========================================
# 1. PREPARAZIONE DEL DATASET
# ==========================================

# Scarichiamo le risorse necessarie da NLTK
# movie_reviews: il dataset con 2000 recensioni (1000 pos e 1000 neg)
# stopwords: le parole comuni da filtrare
# punkt: il modello per la tokenizzazione
nltk.download('movie_reviews')
nltk.download('stopwords')
nltk.download('punkt')

# Il dataset di NLTK non è un DataFrame. Dobbiamo iterare sui file 
# per estrarre il testo grezzo e la categoria (pos/neg) associata.
documents = []
for category in movie_reviews.categories():
    for fileid in movie_reviews.fileids(category):
        documents.append({
            'text': movie_reviews.raw(fileid), # Testo completo della recensione
            'sentiment': category              # Etichetta: 'pos' o 'neg'
        })

# Creiamo un DataFrame Pandas per una gestione più agevole dei dati
df = pd.DataFrame(documents)
print(f"Dataset caricato: {df.shape[0]} recensioni.")

Dataset caricato: 2000 recensioni.


[nltk_data] Downloading package movie_reviews to
[nltk_data]     /Users/antonioserino/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/antonioserino/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/antonioserino/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [107]:
df.head()

,text,sentiment
0,"plot : two teen couples go to a church party ,...",neg
1,the happy bastard's quick movie review \ndamn ...,neg
2,it is movies like these that make a jaded movi...,neg
3,""" quest for camelot "" is warner bros . ' firs...",neg
4,synopsis : a mentally unstable man undergoing ...,neg


In [110]:
print(df['text'][0])

plot : two teen couples go to a church party , drink and then drive . 
they get into an accident . 
one of the guys dies , but his girlfriend continues to see him in her life , and has nightmares . 
what's the deal ? 
watch the movie and " sorta " find out . . . 
critique : a mind-fuck movie for the teen generation that touches on a very cool idea , but presents it in a very bad package . 
which is what makes this review an even harder one to write , since i generally applaud films which attempt to break the mold , mess with your head and such ( lost highway & memento ) , but there are good and bad ways of making all types of films , and these folks just didn't snag this one correctly . 
they seem to have taken this pretty neat concept , but executed it terribly . 
so what are the problems with the movie ? 
well , its main problem is that it's simply too jumbled . 
it starts off " normal " but then downshifts into this " fantasy " world in which you , as an audience member , have no id

In [133]:
df['sentiment'].value_counts()

sentiment
neg    1000
pos    1000
Name: count, dtype: int64

In [ ]:
# ==========================================
# 2. PIPELINE DI PREPROCESSING
# ==========================================

# Carichiamo la lista delle stopword inglesi
stop_words = nltk.corpus.stopwords.words('english')
stemmer = PorterStemmer()

def full_preprocessing(text):
    """
    Questa funzione applica a cascata i passaggi visti a lezione.
    """
    # A. Conversione in minuscolo: standardizza i token (Gatto == gatto)
    text = text.lower()
    
    # B. Pulizia Regex: Rimuoviamo punteggiatura, numeri e caratteri speciali
    # Teniamo solo le lettere (a-z) e gli spazi (\s)
    text = re.sub(r'[^a-z\s]', '', text)
    
    # C. Tokenizzazione: Spezziamo la stringa in una lista di parole
    tokens = text.split()
    
    # D. Rimozione Stopwords: Eliminiamo parole come 'the', 'is', 'at' 
    # che non aggiungono valore alla distinzione del sentiment.
    tokens = [w for w in tokens if w not in stop_words]
    
    # E. STEMMING (Novità!)
    # Riduciamo ogni parola alla sua radice
    tokens = [stemmer.stem(w) for w in tokens]

    #ricordati di ablare gli step di preprocessing e mostrare come variano le performance
    # Torniamo una stringa unica (richiesta dai vettorizzatori di Scikit-Learn)
    return " ".join(tokens)

print("Esecuzione del preprocessing in corso...")
# Applichiamo la funzione a tutta la colonna 'text'
df['clean_text'] = df['text'].apply(full_preprocessing)
print("Preprocessing completato.")

Esecuzione del preprocessing in corso...
Preprocessing completato.


In [135]:
# ==========================================
# 3. SPLIT TRAIN/TEST
# ==========================================

# Dividiamo i dati in due set:
# - Training set (80%): serve per "allenare" il modello
# - Test set (20%): serve per valutarlo su dati che non ha mai visto (simulazione realtà)
# random_state=42 assicura che lo split sia sempre lo stesso ogni volta che eseguiamo il codice
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], 
    df['sentiment'], 
    test_size=0.4, 
    random_state=42
)

In [121]:
y_train

798     neg
1102    pos
105     neg
126     neg
1995    pos
       ... 
1130    pos
1294    pos
860     neg
1459    pos
1126    pos
Name: sentiment, Length: 1200, dtype: object

In [139]:
# ==========================================
# 4. ESPERIMENTO 1: BAG OF WORDS (BoW)
# ==========================================

print("\n--- TEST CON BAG OF WORDS ---")

# Inizializziamo il vettorizzatore. 
# max_features=2000: consideriamo solo le 2000 parole più frequenti nel corpus
# per evitare di avere una matrice troppo "sparsa" e pesante.
bow_vect = CountVectorizer(max_features=20000)

# fit_transform sul train: impara il vocabolario e crea la matrice
X_train_bow = bow_vect.fit_transform(X_train)

# transform sul test: trasforma i dati di test usando SOLO il vocabolario del train
# (Non deve imparare nuove parole dal test, altrimenti bareremmo!)
X_test_bow = bow_vect.transform(X_test)

# Alleniamo una Regressione Logistica
# Un modello lineare semplice ma estremamente potente per la classificazione di testi
model_bow = LogisticRegression(max_iter=1000)
model_bow.fit(X_train_bow, y_train)

# Valutazione
y_pred_bow = model_bow.predict(X_test_bow)
acc_bow = accuracy_score(y_test, y_pred_bow)
precision_bow = classification_report(y_test, y_pred_bow, target_names=['neg', 'pos'])
print(f"Accuratezza BoW: {acc_bow:.4f}")


--- TEST CON BAG OF WORDS ---
Accuratezza BoW: 0.8363


In [141]:
# ==========================================
# 5. ESPERIMENTO 2: TF-IDF
# ==========================================

print("\n--- TEST CON TF-IDF ---")

# Inizializziamo il vettorizzatore TF-IDF con le stesse impostazioni
tfidf_vect = TfidfVectorizer(max_features=20000)

# Applichiamo lo stesso processo di prima
X_train_tfidf = tfidf_vect.fit_transform(X_train)
X_test_tfidf = tfidf_vect.transform(X_test)

# Alleniamo un nuovo modello di Regressione Logistica sui pesi TF-IDF
model_tfidf = LogisticRegression(max_iter=1000)
model_tfidf.fit(X_train_tfidf, y_train)

# Valutazione
y_pred_tfidf = model_tfidf.predict(X_test_tfidf)
acc_tfidf = accuracy_score(y_test, y_pred_tfidf)
print(f"Accuratezza TF-IDF: {acc_tfidf:.4f}")


--- TEST CON TF-IDF ---
Accuratezza TF-IDF: 0.8400


In [138]:
# ==========================================
# 6. CONFRONTO FINALE
# ==========================================

print("\n" + "="*40)
print(f"{'METODO':<15} | {'ACCURATEZZA':<12}")
print("-" * 40)
print(f"{'Bag of Words':<15} | {acc_bow:.4f}")
print(f"{'TF-IDF':<15} | {acc_tfidf:.4f}")
print("="*40)

# Spiegazione per gli studenti
if acc_tfidf > acc_bow:
    print("\n👉 CONCLUSIONE: TF-IDF ha vinto!")
    print("Perché pesare le parole in base alla loro 'rarità' aiuta il modello")
    print("a concentrarsi sui termini davvero descrittivi (es: 'eccellente', 'noioso')")
    print("piuttosto che su quelli semplicemente frequenti.")


METODO          | ACCURATEZZA 
----------------------------------------
Bag of Words    | 0.8363
TF-IDF          | 0.8363


# ESERCIZI FINALI

In [147]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# Download dataset
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_table(url, header=None, names=['label', 'message'])

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

print(f"Dataset caricato: {df.shape[0]} messaggi.")
df.head()

Dataset caricato: 5572 messaggi.


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/antonioserino/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## 1) Label Encoding
### Il modello non capisce "spam" o "ham". Trasforma la colonna label in valori numerici: spam = 1 e ham = 0

# 2) Funzione di Preprocessing
Crea una funzione chiamata clean_text che:

1. Converte in minuscolo.

2. Rimuove la punteggiatura e i numeri.

3. Rimuove le stopword inglesi.

## 3) Stemming

### Aggiorna la funzione precedente aggiungendo lo Stemming di ogni parola prima di restituire la stringa finale.


## 4) Applicazione del Preprocessing
### Applica la funzione clean_text a tutta la colonna message e salva il risultato in una nuova colonna chiamata message_clean.

## 5) Rappresentazione Bag of Words
### Crea una matrice BoW chiamata X_bow usando CountVectorizer, limitando il vocabolario alle 1500 parole più frequenti.

## 6) Rappresentazione TF-IDF
### Crea una matrice TF-IDF chiamata X_tfidf usando TfidfVectorizer (sempre con 1500 parole massime).

## 7) Train/Test Split
### Dividi il dataset TF-IDF in set di allenamento (80%) e test (20%).
### Crea due set di training/test: uno usando i dati BoW e uno usando i dati TF-IDF.

## 8) Classificatore Naive Bayes
### Addestra un naive bayes per bow e uno per tf-idf

In [ ]:
#Aiutino
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train) # Allenamento
y_pred_nb = nb_model.predict(X_test) # Predizione

## 9) Classificatore SVM (Support Vector Machine)
### Addestra una svm per BoW e una per SVM 

In [ ]:
#Aiutino
svm_model = SVC(kernel='linear')
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)

## 10) Valutazione

### stampa le performance relative alla classificazione